# Data-Science Automation (CrewAI)

A **crew of AI agents** (e.g. planner, analyst, modeller) that automates a data-science task on a dataset, dividing the work between specialised roles.

**What it demonstrates**
- Role-based multi-agent design with CrewAI
- Agents that execute code against a pandas DataFrame
- Automating an analysis/modelling pipeline

**Stack:** Python · CrewAI · scikit-learn · pandas


In [3]:
!pip install pywin32

In [1]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

In [2]:
import os
import pandas as pd
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model = "gpt-4.1-mini-2025-04-14", api_key=openai_api_key)


In [11]:
from crewai import LLM
llm = LLM(model = "openai/gpt-4.1-mini-2025-04-14", api_key=openai_api_key)


In [3]:
def print_markdown(text):
    display(Markdown(text))

In [4]:
from notebookExecutor import NotebookCodeExecutor, NotebookCodeExecutorSchema

In [5]:
file_part = "Supplement_Sales_Weekly.csv"
shared_df = pd.read_csv(file_part)


In [7]:
def add_numbers(a,b):
    return a + b

In [8]:
notebook_executor_tool = NotebookCodeExecutor(namespace = globals())

print("✅ Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.")

✅ Custom tool 'NotebookCodeExecutor' instantiated with notebook's global namespace.


In [9]:
test_code = "print(add_numbers(1,3))"

print("\nTesting tool:\n")

print(notebook_executor_tool.run(code = test_code))


Testing tool:

--- Executing Code ---
Code executed successfully. Output:
```output
4

```



In [34]:
# Define the Data Science Planner Agent (no tool needed)
planner_agent = Agent(role = "Lead Data Scientist and Planner",
                      goal = ("Analyze the objective (predict 'Units Sold') assuming data is in a global pandas DataFrame 'shared_df'. "
                              "Create a step-by-step plan for regression analysis. Instruct subsequent agents on the GOALS for each step."
                              "(e.g., inspect data, preprocess, model, evaluate) and tell them to use the 'Notebook Code Executor' tool "
                              "to WRITE and EXECUTE the necessary Python code."),
                    backstory = ("Experienced data scientist planning ML projects. Knows data is in 'shared_df' and agents will write and execute code using a tool."),
                    llm = llm,
                    allow_delegation = False,
                    verbose = True)

In [33]:
# Define the Data Analysis and Preprocessing Agent (needs access to notebook_executer_tool to generate code)
analyst_preprocessor_agent = Agent(role = "Data Analysis and Preprocessing Expert",
                                   goal = (
        "Follow the plan for data analysis and preprocessing. **Write the necessary Python code** using pandas and scikit-learn "
        "to operate on the global pandas DataFrame 'shared_df'. Your code must perform inspection (shape, info, nulls, describe), "
        "handle date/identifiers (convert 'Date', sort, drop 'Date'/'Product Name'), encode categoricals (OneHotEncode 'Platform' modifying 'shared_df'), "
        "and finally **create the global variables X_train, X_test, y_train, y_test** from 'shared_df' using an 80/20 split (shuffle=False). "
        "Use the 'Notebook Code Executor' tool to execute the code you write. Ensure your generated code includes print statements for key results."),
    
                                   backstory = (
        "Meticulous analyst skilled in writing pandas/sklearn code. Uses the 'Notebook Code Executor' tool to run the generated code. "
        "Knows data is in global 'shared_df' and must create global train/test variables."),
                                   llm = llm,
                                   tools = [notebook_executor_tool],  # Assign the custom tool explicitly
                                   allow_delegation = False,
                                   verbose = True)

In [32]:
# Define the Modeling and Evaluation Agent (needs access to notebook_executer_tool to generate code)
modeler_evaluator_agent = Agent(role = "Machine Learning Modeler and Evaluator",
                                goal = (
        "Follow the plan for modeling and evaluation. **Write the necessary Python code** using scikit-learn. "
        "Assume global variables X_train, X_test, y_train, y_test exist. Your code must train a Decision Tree model and a RandomForestRegressor(random_state=42), "
        "make predictions on X_test, calculate and print evaluation metrics (MAE, MSE, RMSE, R²), and print the top 10 feature importances. "
        "Use the 'Notebook Code Executor' tool to execute the code you write. "
        "Finally, include the exact Python code you generated and executed in your final response, formatted in a markdown block."
    ),
                                backstory = (
        "ML engineer specialized in regression. Writes scikit-learn code and uses the 'Notebook Code Executor' tool to run it. "
        "Expects global train/test split variables (X_train etc.) to be available."
    ),
    llm = llm,
    tools = [notebook_executor_tool],  # Assign the custom tool explicitly
    allow_delegation = False,
    verbose = True)


In [15]:
print("✅ CrewAI Agents defined, focusing on code generation.")
print(f"- {planner_agent.role}")
print(f"- {analyst_preprocessor_agent.role} (Tool: {analyst_preprocessor_agent.tools[0].name})")
print(f"- {modeler_evaluator_agent.role} (Tool: {modeler_evaluator_agent.tools[0].name})")

✅ CrewAI Agents defined, focusing on code generation.
- Lead Data Scientist and Planner
- Data Analysis and Preprocessing Expert (Tool: Notebook Code Executor)
- Machine Learning Modeler and Evaluator (Tool: Notebook Code Executor)


In [29]:
# Define the Planning Task (Stays largely the same, instructs agents on GOALS)
planning_task = Task(
    description = (
        "1. Goal: Create a plan for regression predicting 'Units Sold'.\n"
        "2. Data Context: Global pandas DataFrame 'shared_df' is available.\n"
        "3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook Code Executor' tool to WRITE and RUN Python code:\n"
        "    a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).\n"
        "    b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split vars X_train/test, y_train/test [80/20, shuffle=False]).\n"
        "    c. Goal: Train both Decision Tree and Random Forest models using global X_train, y_train (use random_state=42). \n"
        "    d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).\n"
        "    e. Goal: Compare performance between Decision Tree and Random Forest models.\n"
        "    f. Goal: Extract & print top 10 feature importances from the trained model.\n"
        "5. Output: Numbered plan focusing on the objectives for each data science step."
    ),
    expected_output = (
        "Numbered plan outlining the data science goals for subsequent agents, reminding them to generate code and use the 'Notebook Code Executor' tool, interacting with global variables like 'shared_df' and 'X_train'."
    ),
    agent = planner_agent)

In [30]:
# Define the Data Analysis and Preprocessing Task (High-level instructions)
data_analysis_preprocessing_task = Task(
    description = (
        "Follow the analysis/preprocessing plan. Your goal is to inspect and prepare the global 'shared_df' DataFrame and create global training/testing variables. "
        "You MUST **generate Python code** to achieve this and then execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Inspect the 'shared_df' DataFrame (print shape, info(), isnull().sum(), describe()).\n"
        "2. Convert 'Date' column in 'shared_df' to datetime objects, sort 'shared_df' by 'Date', then drop the 'Date' and 'Product Name' columns from 'shared_df'.\n"
        "3. One-Hot Encode the 'Platform' column in 'shared_df' (use pd.get_dummies, drop_first=True). **Crucially, ensure 'shared_df' DataFrame variable is updated with the result of the encoding.**\n"
        "4. Create a global variable 'y' containing the 'Units Sold' column from 'shared_df'.\n"
        "5. Create a global variable 'X' containing the remaining columns from the updated 'shared_df' (after dropping 'Units Sold').\n"
        "6. Split 'X' and 'y' into global variables: 'X_train', 'X_test', 'y_train', 'y_test' using an 80/20 split with `shuffle=False`. Ensure these four variables are created in the global scope.\n"
        "Make sure your generated code includes necessary imports (like pandas, train_test_split) and print statements for verification (e.g., printing shapes of created variables like X_train.shape)."
        # "Remember to pass the required libraries (e.g., ['pandas', 'scikit-learn']) to the tool if your code uses them, although they should be pre-imported in this notebook." # Optional hint, often the agent figures out imports
    ),
    expected_output = (
        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code. This includes printouts confirming:\n"
        "- Initial data inspection results for 'shared_df'.\n"
        "- Confirmation of DataFrame modifications (e.g., shape after encoding).\n"
        "- Confirmation of the creation and shapes of global variables X, y, X_train, X_test, y_train, y_test."
    ),
    agent = analyst_preprocessor_agent,
    tools = [notebook_executor_tool],  # Explicitly list tool
)

In [42]:
# Define the Modeling and Evaluation Task (High-level instructions)
modeling_evaluation_task = Task(
    description = (
        "Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results. "
        "You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and execute it using the 'Notebook Code Executor' tool. "
        "Specifically, your generated code needs to:\n"
        "1. Train a `DecisionTreeRegressor` model (use `random_state=42`) using the global `X_train` and `y_train` variables. Store the trained model in a global variable named `dt_model`.\n"
        "2. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train` variables. Store the trained model in a global variable named `rf_model`.\n"
        "3. Make predictions on the global `X_test` variable.\n"
        "4. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global `y_test` variable.\n"
        "5. Compare the performance of both models and highlight which one performs better and why.\n"
        "6. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for feature names).\n"
        "Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from sklearn.metrics, numpy, pandas) and print statements for all results.\n"
        "Finally, include the exact Python code you generated and executed within a markdown code block (```python...```) in your final response."
        # "Remember to pass required libraries like ['scikit-learn', 'pandas', 'numpy'] to the tool if needed." # Optional hint
    ),
#    expected_output = (
#        "Output from the 'Notebook Code Executor' tool showing the successful execution of agent-generated code, including:\n"
#        "- Printed regression metrics (MAE, MSE, RMSE, R²).\n"
#        "- Printed top 10 feature importances.\n"
#        "The final response MUST also contain a markdown code block (```python...```) showing the exact Python code that was generated and executed for these steps."
#    ),

    expected_output = (
        "A markdown report with TWO clearly separated sections, in this exact order:\n\n"
        "## Results\n"
        "The actual numerical output returned by the Notebook Code Executor tool:\n"
        "- Regression metrics (MAE, MSE, RMSE, R²) for both the Decision Tree and Random Forest models.\n"
        "- A statement of which model is better based on R².\n"
        "- The top 10 feature importances.\n\n"
        "## Code\n"
        "A single ```python ... ``` block containing the exact Python code that was generated and executed.\n\n"
        "Both sections are REQUIRED. The Results section must show the real executed numbers, "
        "not placeholders, and must appear BEFORE the Code section."
),
    agent = modeler_evaluator_agent,
    tools = [notebook_executor_tool],  # Explicitly list tool
)

print("✅ CrewAI Tasks defined with high-level instructions for code generation.")

✅ CrewAI Tasks defined with high-level instructions for code generation.


In [38]:
# Let's Create the Crew
regression_crew = Crew(
    agents = [planner_agent, analyst_preprocessor_agent, modeler_evaluator_agent],
    tasks = [planning_task, data_analysis_preprocessing_task, modeling_evaluation_task],
    process = Process.sequential,
    verbose = 1,  # Use detailed output to see agent thoughts and tool usage
    output_log_file = True)

In [43]:

print("Starting the Crew execution (Agents will generate code)...")

crew_result = await regression_crew.kickoff_async()

Starting the Crew execution (Agents will generate code)...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c0168e09-162b-4178-aad4-e4d12cd052a2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train both Decision Tree and Random Forest models using global X_train, y_train (use              │
│  random_state=42).                                                                                              │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Compare performance between Decision Tree and Random Forest models.                               │
│      f. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│  ID: 4755c0b7-26ac-4363-b819-bea313c29ad8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Scientist and Planner                                                                         │
│                                                                                                                 │
│  Task: 1. Goal: Create a plan for regression predicting 'Units Sold'.                                           │
│  2. Data Context: Global pandas DataFrame 'shared_df' is available.                                             │
│  3. Plan Steps: Outline sequence, instructing agents on their GOALS for each step and to use the 'Notebook      │
│  Code Executor' tool to WRITE and RUN Python code:                                                              │
│      a. Goal: Inspect global 'shared_df' (shape, info, nulls, describe).                                        │
│      b. Goal: Preprocess global 'shared_df' (handle Date [to_datetime, sort, drop], drop identifiers ['Product  │
│  Name'], OneHotEncode 'Platform' [update 'shared_df'], create global X/y vars, create global train/test split   │
│  vars X_train/test, y_train/test [80/20, shuffle=False]).                                                       │
│      c. Goal: Train both Decision Tree and Random Forest models using global X_train, y_train (use              │
│  random_state=42).                                                                                              │
│      d. Goal: Evaluate model on global X_test (predict, calc & print MAE, MSE, RMSE, R2).                       │
│      e. Goal: Compare performance between Decision Tree and Random Forest models.                               │
│      f. Goal: Extract & print top 10 feature importances from the trained model.                                │
│  5. Output: Numbered plan focusing on the objectives for each data science step.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  from sklearn.tree import DecisionTreeRegressor                                                                 │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│                                                                                                                 │
│  # Train Decision Tree model                                                                                    │
│  print("Training Decision Tree Regressor...")                                                                   │
│  dt_model = DecisionTreeRegressor(random_state=42)                                                              │
│  dt_model.fit(X_train, y_train)                                                                                 │
│                                                                                                                 │
│  # Train Random Forest model                                                                                    │
│  print("Training Random Forest Regressor...")                                                                   │
│  rf_model = RandomForestRegressor(random_state=42)                                                              │
│  rf_model.fit(X_train, y_train)                                                                                 │
│                                                                                                                 │
│  # Predictions                                                                                                  │
│  print("Making predictions on test set...")                                                                     │
│  dt_pred = dt_model.predict(X_test)                                                                             │
│  rf_pred = rf_model.predict(X_test)                                                                             │
│                                                                                                                 │
│  # Calculate and print evaluation metrics function                                                              │
│  def print_metrics(y_true, y_pred, model_name):                                                                 │
│      mae = mean_absolute_error(y_true, y_pred)                                                                  │
│      mse = mean_squared_error(y_true, y_pred)                                                                   │
│      rmse = np.sqrt(mse)                                                                                        │
│      r2 = r2_score(y_true, y_pred)                                                                              │
│      print(f"{model_name} Performance Metrics:")                                                                │
│      print(f"  Mean Absolute Error (MAE): {mae:.4f}")  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Follow the modeling/evaluation plan. Your goal is to train a model, evaluate it, and report results.     │
│  You MUST **generate Python code** assuming global variables X_train, X_test, y_train, y_test exist, and        │
│  execute it using the 'Notebook Code Executor' tool. Specifically, your generated code needs to:                │
│  1. Train a `DecisionTreeRegressor` model (use `random_state=42`) using the global `X_train` and `y_train`      │
│  variables. Store the trained model in a global variable named `dt_model`.                                      │
│  2. Train a `RandomForestRegressor` model (use `random_state=42`) using the global `X_train` and `y_train`      │
│  variables. Store the trained model in a global variable named `rf_model`.                                      │
│  3. Make predictions on the global `X_test` variable.                                                           │
│  4. Calculate and print the MAE, MSE, RMSE, and R-squared metrics by comparing predictions against the global   │
│  `y_test` variable.                                                                                             │
│  5. Compare the performance of both models and highlight which one performs better and why.                     │
│  6. Calculate and print the top 10 feature importances from the trained model (using `X_train.columns` for      │
│  feature names).                                                                                                │
│  Make sure your generated code includes necessary imports (like RandomForestRegressor, metrics functions from   │
│  sklearn.metrics, numpy, pandas) and print statements for all results.                                          │
│  Finally, include the exact Python code you generated and executed within a markdown code block                 │
│  (```python...```) in your final response.                                                                      │
│  Agent: Machine Learning Modeler and Evaluator                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c0168e09-162b-4178-aad4-e4d12cd052a2                                                                       │
│  Final Output: ```python                                                                                        │
│  from sklearn.tree import DecisionTreeRegressor                                                                 │
│  from sklearn.ensemble import RandomForestRegressor                                                             │
│  from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score                                  │
│  import numpy as np                                                                                             │
│  import pandas as pd                                                                                            │
│                                                                                                                 │
│  # Train Decision Tree model                                                                                    │
│  print("Training Decision Tree Regressor...")                                                                   │
│  dt_model = DecisionTreeRegressor(random_state=42)                                                              │
│  dt_model.fit(X_train, y_train)                                                                                 │
│                                                                                                                 │
│  # Train Random Forest model                                                                                    │
│  print("Training Random Forest Regressor...")                                                                   │
│  rf_model = RandomForestRegressor(random_state=42)                                                              │
│  rf_model.fit(X_train, y_train)                                                                                 │
│                                                                                                                 │
│  # Predictions                                                                                                  │
│  print("Making predictions on test set...")                                                                     │
│  dt_pred = dt_model.predict(X_test)                                                                             │
│  rf_pred = rf_model.predict(X_test)                                                                             │
│                                                                                                                 │
│  # Calculate and print evaluation metrics function                                                              │
│  def print_metrics(y_true, y_pred, model_name):                                                                 │
│      mae = mean_absolute_error(y_true, y_pred)                                                                  │
│      mse = mean_squared_error(y_true, y_pred)                                                                   │
│      rmse = np.sqrt(mse)                                                                                        │
│      r2 = r2_score(y_true, y_pred)                                                                              │
│      print(f"{model_name} Performance Metrics:")                                                                │
│      print(f"  Mean Absolute Error (MAE): {mae:.4f}") 

┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└───────────────────────────────────────

In [45]:
print("\n\n🏁 Crew execution finished.")
print("\nCrew Final Result (Output of last task):")
print("========================================")

print_markdown(crew_result.raw)



🏁 Crew execution finished.

Crew Final Result (Output of last task):


```python
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Train Decision Tree model
print("Training Decision Tree Regressor...")
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

# Train Random Forest model
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
print("Making predictions on test set...")
dt_pred = dt_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

# Calculate and print evaluation metrics function
def print_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Performance Metrics:")
    print(f"  Mean Absolute Error (MAE): {mae:.4f}")
    print(f"  Mean Squared Error (MSE): {mse:.4f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"  R-squared (R2): {r2:.4f}")
    print()
    return mae, mse, rmse, r2

# Evaluate Decision Tree
print("Evaluating Decision Tree Model:")
dt_metrics = print_metrics(y_test, dt_pred, "Decision Tree")

# Evaluate Random Forest
print("Evaluating Random Forest Model:")
rf_metrics = print_metrics(y_test, rf_pred, "Random Forest")

# Compare performance
print("Comparison of Model Performance:")
if rf_metrics[3] > dt_metrics[3]:
    print("Random Forest performs better in terms of R-squared (higher is better).")
else:
    print("Decision Tree performs better in terms of R-squared (higher is better).")

if rf_metrics[0] < dt_metrics[0]:
    print("Random Forest performs better in terms of MAE (lower is better).")
else:
    print("Decision Tree performs better in terms of MAE (lower is better).")

# Assuming Random Forest is better, print top 10 feature importances
print("\nTop 10 Feature Importances from the Random Forest Model:")
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top10_features = importances.sort_values(ascending=False).head(10)
for i, (feature, importance) in enumerate(top10_features.items(), 1):
    print(f" {i}. {feature}: {importance:.4f}")
```

Output from execution:

Training Decision Tree Regressor...
Training Random Forest Regressor...
Making predictions on test set...
Evaluating Decision Tree Model:
Decision Tree Performance Metrics:
  Mean Absolute Error (MAE): 195.7799
  Mean Squared Error (MSE): 125822.8883
  Root Mean Squared Error (RMSE): 354.7152
  R-squared (R2): 0.6651

Evaluating Random Forest Model:
Random Forest Performance Metrics:
  Mean Absolute Error (MAE): 160.0538
  Mean Squared Error (MSE): 82288.8925
  Root Mean Squared Error (RMSE): 286.8604
  R-squared (R2): 0.7810

Comparison of Model Performance:
Random Forest performs better in terms of R-squared (higher is better).
Random Forest performs better in terms of MAE (lower is better).

Top 10 Feature Importances from the Random Forest Model:
 1. Price: 0.7839
 2. Discount: 0.1101
 3. Category_Protein: 0.0260
 4. Category_Performance: 0.0145
 5. Category_Fat Burner: 0.0118
 6. Category_Mineral: 0.0091
 7. Walmart: 0.0089
 8. Location_UK: 0.0073
 9. iHerb: 0.0072
10. Location_USA: 0.0062
